# 🧪 프롬프트 최적화 루프

기존 평가 데이터를 활용해 **Canopy 요약 프롬프트를 체계적으로 개선**합니다.

```
Register Prompt  →  Run Queries  →  Attach Feedback
       ↑                                    ↓
  New Version  ←  Optimize  ←  Evaluate  ←  Build Dataset
```

**여기서 할 일:**
1. MLflow `eval` 데이터셋(test 워크스페이스)과 `summary_tests.yaml`을 합쳐 학습 데이터셋을 **구성**합니다
2. 최신 요약(Summarization) 프롬프트를 **불러와 테스트**합니다
3. MLflow의 GEPA가 점수가 낮은 사례를 바탕으로 프롬프트를 자동으로 다시 작성하도록 **최적화**합니다
4. v1과 최적화된 버전을 나란히 **비교**하고 승격할지 결정합니다(이 단계는 노트북을 마친 뒤 MLflow에서 진행합니다)

---
## 0. 설정

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import yaml
from typing import Literal

import mlflow
from mlflow import MlflowClient
from mlflow.genai.scorers import scorer
from mlflow.genai.optimize import GepaPromptOptimizer
from mlflow.genai.judges import make_judge
from openai import OpenAI
import pandas as pd

> ⚠️ **참고:** 아래 변수를 본인의 클러스터와 모델 엔드포인트에 맞게 수정하세요.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# 설정 — 본인의 환경에 맞게 수정하세요
# ─────────────────────────────────────────────────────────────────

# MLflow 서버
MLFLOW_TRACKING_URI = "https://mlflow.redhat-ods-applications.svc.cluster.local:8443"

os.environ["MLFLOW_TRACKING_AUTH"] = "kubernetes"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

NAMESPACE_PATH = "/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(NAMESPACE_PATH):
    with open(NAMESPACE_PATH) as f:
        username = f.read().strip().split("-")[0]
        os.environ["MLFLOW_WORKSPACE"] = f"{username}-toolings"

SA_TOKEN_PATH = "/run/secrets/kubernetes.io/serviceaccount/token"
if os.path.exists(SA_TOKEN_PATH):
    with open(SA_TOKEN_PATH) as f:
        os.environ["MLFLOW_TRACKING_TOKEN"] = f.read().strip()

TOOLINGS_WORKSPACE = os.environ.get("MLFLOW_WORKSPACE", "")
TEST_WORKSPACE     = TOOLINGS_WORKSPACE.replace("-toolings", "-test")
print(f"Toolings workspace: {TOOLINGS_WORKSPACE}")
print(f"Test workspace:     {TEST_WORKSPACE}")

# MLflow 이름
EXPERIMENT_NAME = "summarization"
PROMPT_NAME     = "summarization"

# LLM — Canopy가 사용하는 것과 같은 모델을 지정
LLM_API_KEY  = "fake"
LLM_BASE_URL = "http://llama-32-predictor.ai501.svc.cluster.local:8080/v1"
LLM_MODEL    = "llama32"
if LLM_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = LLM_BASE_URL
os.environ["OPENAI_API_KEY"] = LLM_API_KEY

# 요약 테스트 설정 파일 경로(다른 위치에 마운트되어 있다면 수정)
YAML_PATH         = "../../evals/summarization/summary_tests.yaml"
JUDGE_PROMPT_PATH = "../../evals/summarization/judge_prompt.txt"

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"✓ Connected to MLflow — workspace: {TOOLINGS_WORKSPACE}, experiment: {EXPERIMENT_NAME}")

mlflow.openai.autolog()
llm_client = OpenAI(
    api_key=LLM_API_KEY,
    **(dict(base_url=LLM_BASE_URL) if LLM_BASE_URL else {}),
)

def _switch_workspace(workspace):
    os.environ["MLFLOW_WORKSPACE"] = workspace
    mlflow.set_tracking_uri("")
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# 최신 프롬프트 버전 확인(프롬프트는 toolings 워크스페이스에 있음)
existing_prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
PROMPT_V1_URI = f"prompts:/{PROMPT_NAME}/{existing_prompt.version}"
print(f"✓ Using prompt '{PROMPT_NAME}' v{existing_prompt.version}  ({PROMPT_V1_URI})")
print(f"\n── Prompt template ──\n{existing_prompt.template}\n─────────────────────")

---
## 2단계: 통합 평가 데이터셋 구성하기

옵티마이저가 사용할 더 풍부한 학습 데이터셋을 만들기 위해 두 가지 소스를 합칩니다.

1. **MLflow `eval` 데이터셋**: 현재 워크스페이스의 `summarization` experiment에 이미 저장된 레코드로, 이전 파이프라인 실행에서 누적된 것입니다.
2. **`summary_tests.yaml`**: prompts 저장소에 있는 구조화된 테스트 케이스입니다.

In [ ]:
# test 워크스페이스에서 기존 eval 데이터셋 불러오기
_switch_workspace(TEST_WORKSPACE)
try:
    mlflow_dataset = mlflow.genai.get_dataset(name="eval")
    mlflow_records = mlflow_dataset.to_dict().get("records", [])
    print(f"✓ Loaded MLflow 'eval' dataset from {TEST_WORKSPACE}: {len(mlflow_records)} record(s)")
    for r in mlflow_records[:2]:
        msgs = r.get("inputs", {}).get("messages", [])
        preview = msgs[0]["content"][:70] if msgs else str(r.get("inputs", ""))[:70]
        print(f"  - {preview}...")
except Exception as e:
    print(f"⚠ Could not load 'eval' dataset: {e}")
    mlflow_records = []
finally:
    _switch_workspace(TOOLINGS_WORKSPACE)

In [ ]:
# summary_tests.yaml에서 테스트 케이스 불러오기
with open(YAML_PATH) as f:
    test_config = yaml.safe_load(f)

yaml_records = []
for test in test_config.get("tests", []):
    # session_id 제거 — 백엔드 라우팅 키이며 judge와는 관련 없음
    inputs = {k: v for k, v in test["inputs"].items() if k != "session_id"}
    yaml_records.append({
        "inputs":       inputs,
        "expectations": test.get("expectations", {}),
    })

print(f"✓ Loaded {len(yaml_records)} record(s) from summary_tests.yaml")
for r in yaml_records:
    msgs = r["inputs"].get("messages", [])
    preview = msgs[0]["content"][:70] if msgs else ""
    print(f"  - {preview}...")

In [ ]:
# 두 소스 합치기 — 각 레코드에서 inputs와 expectations만 유지
combined_records = []
for r in mlflow_records:
    combined_records.append({
        "inputs":       r.get("inputs", {}),
        "expectations": r.get("expectations", {}),
    })
combined_records.extend(yaml_records)

print(f"✓ Combined training data: {len(combined_records)} record(s) total")
print(f"  ({len(mlflow_records)} from MLflow eval + {len(yaml_records)} from YAML)")

---
## 3단계: Scorer 정의하기

평가 파이프라인에서 사용하는 것과 같은 `summary_quality` LLM judge를 재사용합니다.
judge 지침은 테스트 설정과 함께 있는 `judge_prompt.txt`에서 불러옵니다.

judge는 입력 메시지, 생성된 요약, `expected_result` 참고 답안을 받은 뒤
**"yes"**(좋은 요약) 또는 **"no"**(품질 기준 미달)로 답합니다.

In [ ]:
with open(JUDGE_PROMPT_PATH) as f:
    judge_instructions = f.read()

summary_quality = make_judge(
    name="summary_quality",
    instructions=judge_instructions,
    feedback_value_type=Literal["yes", "no"],
    model=f"openai:/{LLM_MODEL}",
    base_url=LLM_BASE_URL + "/chat/completions",
    extra_headers={"Authorization": "Bearer no-key-required"},
)

@scorer
def is_shorter(outputs: str, inputs: dict) -> bool:
    """요약이 입력 텍스트보다 짧은가?"""
    user_content = " ".join(
        m.get("content", "") for m in inputs.get("messages", []) if m.get("role") == "user"
    )
    return len(outputs) < len(user_content)

print("✓ Scorers defined: summary_quality, is_shorter")
print(f"  Judge model: openai:/{LLM_MODEL}")
print(f"  Judge prompt: {JUDGE_PROMPT_PATH}")

---
## 4단계: 기준(Baseline) 평가

최적화 전에 현재 프롬프트로 통합 데이터셋을 평가해 둡니다. 그러면 자동 파이프라인이 새 프롬프트 버전을 가져갔을 때 비교할 점수가 생깁니다.

In [ ]:
def predict_v1(messages, **kwargs) -> str:
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)
    user_content = " ".join(m["content"] for m in messages if m.get("role") == "user")
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": user_content},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Running baseline evaluation with prompt v{existing_prompt.version}...")
with mlflow.start_run(run_name="summary_tests_baseline"):
    eval_baseline = mlflow.genai.evaluate(
        data=combined_records,
        predict_fn=predict_v1,
        scorers=[summary_quality, is_shorter],
    )

print("\n── Baseline scores ──")
for metric, value in sorted(eval_baseline.metrics.items()):
    print(f"  {metric:<35} {f'{value:.0%}' if isinstance(value, float) else value}")

---
## 5단계: GEPA로 최적화하기

**GEPA(Generate, Evaluate, Predict, Adapt)**는 MLflow의 자동 프롬프트 옵티마이저입니다.

작동 방식:
1. 학습 데이터에 대해 `predict_fn`을 실행하고 각 출력을 `summary_quality`로 채점합니다
2. *reflection 모델*을 사용해 어떤 예시가 왜 실패했는지 분석합니다
3. 이러한 약점을 보완하도록 프롬프트를 다시 작성합니다
4. 개선된 프롬프트를 Prompt Registry에 새 버전으로 등록합니다

> ⏱️ 몇 분 정도 걸립니다. 옵티마이저가 내부적으로 여러 차례 평가를 실행합니다.

참고: 시간 초과가 발생하면 `max_metric_calls` 값을 바꿔 보세요.

In [ ]:
%pip install -q 'gepa>=0.0.26'

In [ ]:
# GEPA 실행 전에 autologging을 끕니다. autolog는 OpenAI API trace 전체를 기록하는데,
# GEPA가 이를 reflection 프롬프트에 그대로 포함하므로 모델의 컨텍스트 윈도를 쉽게 초과합니다.
mlflow.openai.autolog(disable=True)


def predict_fn(messages, **kwargs) -> str:
    """
    옵티마이저용 predict 함수입니다.
    옵티마이저가 호출을 가로채 어떤 템플릿을 다시 작성할지 식별할 수 있도록
    mlflow.genai.load_prompt()를 반드시 여기에서 호출해야 합니다.
    """
    prompt = mlflow.genai.load_prompt(PROMPT_V1_URI)  # ← 옵티마이저가 여기에 개입함
    user_content = " ".join(m["content"] for m in messages if m.get("role") == "user")
    response = llm_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": prompt.template},
            {"role": "user",   "content": user_content},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content


print(f"Optimizing '{PROMPT_NAME}' using {LLM_MODEL} as the reflection model...")
print("This may take a few minutes as the optimizer runs multiple passes.\n")

# max_metric_calls는 GEPA가 수행하는 전체 scorer 평가 횟수를 제어합니다(기본값: 100).
# 값이 낮을수록 빨리 끝나며, 옵티마이저가 더 많은 후보를 탐색하게 하려면 값을 높이세요.
opt_result = mlflow.genai.optimize_prompts(
    predict_fn=predict_fn,
    train_data=combined_records,
    prompt_uris=[PROMPT_V1_URI],
    optimizer=GepaPromptOptimizer(reflection_model=f"openai:/{LLM_MODEL}", max_metric_calls=30),
    scorers=[summary_quality, is_shorter],
)

OPTIMIZED_PROMPT_URI = opt_result.optimized_prompts[0].uri
optimized_version    = opt_result.optimized_prompts[0].version

In [ ]:
print(f"\n✓ Optimization complete!")
print(f"  → New version: v{optimized_version}")

optimized_prompt = mlflow.genai.load_prompt(OPTIMIZED_PROMPT_URI)
print(f"\nOptimized prompt:\n{'─'*50}")
print(optimized_prompt.template)
print("─" * 50)

---
## 다음 단계는?

최적화된 프롬프트의 점수가 더 높다면 test 환경으로, 그다음에는 프로덕션 Canopy로 승격해 보세요! <USER_NAME>-canopy 환경의 Prompts에서 최적화된 프롬프트를 확인해 보세요! 

결과가 마음에 든다면 <USER_NAME>-tooling 환경으로 옮겨도 좋습니다!

어느 쪽이든, 이제 피드백 루프가 완전히 닫혔습니다: 실제 사용자 신호 → 평가 데이터셋 → 자동 최적화 → 더 나은 Canopy.